# MWAMNet — Stage 5: the explainability figures

In [ ]:
#@title 1 - Setup
!pip -q install kagglehub shap

import os, glob, gc, math, json
import numpy as np
import tensorflow as tf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

for _g in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(_g, True)
    except Exception as e:
        print("memory-growth setting skipped:", e)

CFG = dict(
    KAGGLE_ID = "alik05/forest-fire-dataset",
    IMG       = 224,
    SEED      = 42,
    VAL_FRAC  = 0.20,
    EPOCHS    = 60,
    BATCH     = 32,
    PATIENCE  = 10,
    EXTRACT_BS= 16,
    L2        = 0.01,
    DROPOUT   = 0.5,
    LR        = 1e-4,
    WD        = 1e-4,
)
S   = CFG["IMG"]
OUT = "/content/MWAMNet_figs"
os.makedirs(OUT, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300,
    "font.size": 10, "axes.titlesize": 10, "axes.labelsize": 10,
    "savefig.bbox": "tight", "pdf.fonttype": 42,
})

def oshape(layer_or_model):
    # Keras 3 removed .output_shape; Keras 2 has no .output on some layers.

    for get in (lambda: tuple(layer_or_model.output.shape),
                lambda: tuple(layer_or_model.output_shape)):
        try:
            v = get()
            if v: return v
        except Exception:
            pass
    return None

import keras as _keras
print("TensorFlow:", tf.__version__, "| Keras:", _keras.__version__)
print("GPU       :", tf.config.list_physical_devices("GPU") or "NONE  <-- switch to T4 GPU")

TensorFlow: 2.20.0 | Keras: 3.13.2
GPU       : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2 · The dataset and the held-out split

Identical to Stage 1: the same seed, the same stratified 80/20 split of the
`Training` folder, and the same 380-image `Testing` folder held out entirely. The
figures therefore describe the same model the tables describe.

In [ ]:
#@title 2 - Download the dataset and rebuild the split
import kagglehub
from sklearn.model_selection import train_test_split

root = kagglehub.dataset_download(CFG["KAGGLE_ID"])
base = glob.glob(os.path.join(root, "**", "Forest Fire Dataset"), recursive=True)[0]
TRAIN_DIR, TEST_DIR = os.path.join(base, "Training"), os.path.join(base, "Testing")

def load(paths):
    a = np.zeros((len(paths), S, S, 3), dtype=np.uint8)
    for i, p in enumerate(paths):
        with Image.open(p) as im:
            a[i] = np.asarray(im.convert("RGB").resize((S, S), Image.BILINEAR))
    return a

pool_paths, pool_y = [], []
for cls, lab in (("nofire", 0), ("fire", 1)):
    fs = sorted(glob.glob(os.path.join(TRAIN_DIR, cls, "*.jpg")))
    pool_paths += fs; pool_y += [lab] * len(fs)
pool_y = np.array(pool_y, dtype=np.int32)

test_paths = sorted(glob.glob(os.path.join(TEST_DIR, "*.jpg")))
test_y = np.array([0 if os.path.basename(p).lower().startswith("nofire") else 1
                   for p in test_paths], dtype=np.int32)

tr_p, va_p, y_tr, y_va = train_test_split(
    pool_paths, pool_y, test_size=CFG["VAL_FRAC"],
    random_state=CFG["SEED"], stratify=pool_y)

X_tr, X_va, X_te = load(tr_p), load(va_p), load(test_paths)
y_te = test_y

print("train %4d   val %4d   TEST %4d  (held out)" % (len(y_tr), len(y_va), len(y_te)))

100%|██████████| 142M/142M [00:01<00:00, 118MB/s]

Extracting files...


train 1216   val  304   TEST  380  (held out)


In [ ]:
#@title 3 - Print every conv layer with its output shape
from tensorflow.keras.applications import DenseNet201, MobileNetV3Large
from tensorflow.keras.applications import densenet, mobilenet_v3

BACKBONES = {
    "DenseNet201":      (DenseNet201,      densenet.preprocess_input),
    "MobileNetV3Large": (MobileNetV3Large, mobilenet_v3.preprocess_input),
}

diagnosis = {}
nets = {}
for name, (Base, prep) in BACKBONES.items():
    net = Base(weights="imagenet", include_top=False, input_shape=(S, S, 3))
    net.trainable = False
    nets[name] = net

    convs = [l for l in net.layers
             if "conv" in l.name.lower()
             and oshape(l) is not None and len(oshape(l)) == 4]
    if not convs:
        raise RuntimeError("no 4-D 'conv' layers found in %s - report this" % name)
    last  = convs[-1]
    print("=" * 78)
    print(name, " - %d layers with 'conv' in the name" % len(convs))
    print("=" * 78)
    print("%-5s %-38s %-18s %s" % ("idx", "layer name", "output shape", ""))
    show = list(range(0, 3)) + list(range(len(convs) - 3, len(convs)))
    for i in sorted(set(show)):
        l = convs[i]
        tag = ""
        if i == 0: tag = "<-- your [0]"
        elif i == 1: tag = "<-- your [1]"
        elif i == len(convs) - 1: tag = "<-- CORRECT (last conv)"
        print("%-5d %-38s %-18s %s" % (i, l.name, str(oshape(l)[1:]), tag))
        if i == 2 and len(convs) > 6: print("      ...")
    diagnosis[name] = dict(
        n_conv        = len(convs),
        first_name    = convs[0].name,
        first_spatial = int(oshape(convs[0])[1]),
        last_name     = last.name,
        last_spatial  = int(oshape(last)[1]),
        backbone_out  = tuple(int(v) for v in oshape(net)[1:]),
    )
    print()

print("=" * 78)
print("VERDICT")
print("=" * 78)
for name, d in diagnosis.items():
    bad = d["first_spatial"] > 4 * d["last_spatial"]
    print("%-18s first conv is %dx%d, last conv is %dx%d" % (
        name, d["first_spatial"], d["first_spatial"], d["last_spatial"], d["last_spatial"]))
    print("%-18s %s" % ("",
        "Indexing [0] therefore gave you an EARLY layer. The submitted heatmaps are"
        " edge texture, not object evidence - regenerate them." if bad else
        "Indexing [0] happened to land near the end - check the figure by eye."))
print()
print("The backbone output used below for Grad-CAM:")
for name, d in diagnosis.items():
    print("   %-18s %s   (%dx%d spatial - this is what a heatmap should come from)"
          % (name, d["backbone_out"], d["backbone_out"][0], d["backbone_out"][1]))

with open(os.path.join(OUT, "gradcam_layer_diagnosis.json"), "w") as f:
    json.dump(diagnosis, f, indent=2)

74836368/74836368 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
DenseNet201  - 692 layers with 'conv' in the name
idx   layer name                             output shape       
0     conv1_conv                             (112, 112, 64)     <-- your [0]
1     conv1_bn                               (112, 112, 64)     <-- your [1]
2     conv1_relu                             (112, 112, 64)     
      ...
689   conv5_block32_1_relu                   (7, 7, 128)        
690   conv5_block32_2_conv                   (7, 7, 32)         
691   conv5_block32_concat                   (7, 7, 1920)       <-- CORRECT (last conv)

12683000/12683000 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
MobileNetV3Large  - 146 layers with 'conv' in the name
idx   layer name                             output shape       
0     conv                                   (112, 112, 16)     <-- your [0]
1     conv_bn                                (112, 112, 16)     <-- your [1]
2     expanded_conv_depthwise                (112, 112, 16)  

In [ ]:
#@title 4 - Extract features and train the classifier head
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import (Dense, BatchNormalization, Dropout,
                                     Concatenate, GlobalAveragePooling2D, Lambda)
from tensorflow.keras.regularizers import l2 as L2reg

def extract(net, prep, u8, bs=None):
    bs  = bs or CFG["EXTRACT_BS"]
    pool = GlobalAveragePooling2D()
    out = np.empty((len(u8), int(oshape(net)[-1])), dtype=np.float32)
    i = 0
    while i < len(u8):
        try:
            b = tf.cast(u8[i:i + bs], tf.float32)
            out[i:i + int(b.shape[0])] = pool(net(prep(b), training=False)).numpy()
            i += bs
        except tf.errors.ResourceExhaustedError:
            if bs == 1: raise
            bs = max(1, bs // 2); gc.collect()
            print("   GPU out of memory - retrying at batch size %d" % bs)
    return out

FEATS = {}
for name, (Base, prep) in BACKBONES.items():
    net = nets[name]
    print("extracting %s ..." % name)
    FEATS[name] = dict(
        train = extract(net, prep, X_tr),
        val   = extract(net, prep, X_va),
        test  = extract(net, prep, X_te),
    )
    gc.collect()
print("feature dimensions:", {k: v["train"].shape[1] for k, v in FEATS.items()})

BB = ["DenseNet201", "MobileNetV3Large"]
dims = [FEATS[b]["train"].shape[1] for b in BB]

def build_head(dims, l2v, drop, units=(1024, 512)):
    ins = [Input(shape=(d,), name="feat_%d" % i) for i, d in enumerate(dims)]
    br  = [Dropout(drop)(BatchNormalization()(i)) for i in ins]
    x   = Concatenate()(br) if len(br) > 1 else br[0]
    for u in units:
        x = Dense(u, activation="relu", kernel_regularizer=L2reg(l2v))(x)
        x = BatchNormalization()(x)
        x = Dropout(drop)(x)
    return Model(ins, Dense(1, activation="sigmoid")(x), name="mwamnet_head")

tf.keras.utils.set_random_seed(CFG["SEED"])
head = build_head(dims, CFG["L2"], CFG["DROPOUT"])

Xtr = [FEATS[b]["train"] for b in BB]
Xva = [FEATS[b]["val"]   for b in BB]
Xte = [FEATS[b]["test"]  for b in BB]

steps = math.ceil(len(y_tr) / CFG["BATCH"]) * CFG["EPOCHS"]
lr = tf.keras.optimizers.schedules.CosineDecay(CFG["LR"], steps, alpha=1e-6)
head.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=lr,
                                                 weight_decay=CFG["WD"]),
             loss="binary_crossentropy", metrics=["accuracy"])
head.fit(Xtr, y_tr, validation_data=(Xva, y_va),
         epochs=CFG["EPOCHS"], batch_size=CFG["BATCH"], verbose=0,
         callbacks=[tf.keras.callbacks.EarlyStopping(
             monitor="val_accuracy", patience=CFG["PATIENCE"],
             restore_best_weights=True, mode="max")])

p_te = head.predict(Xte, verbose=0).ravel()
acc  = float(((p_te > 0.5).astype(int) == y_te).mean())
print("held-out test accuracy: %.4f   (Table 3 reports 0.9895)" % acc)

extracting DenseNet201 ...
extracting MobileNetV3Large ...
feature dimensions: {'DenseNet201': 1920, 'MobileNetV3Large': 960}
held-out test accuracy: 0.9895   (Table 3 reports 0.9895)


## 5 · Wire the trained head onto the backbones

The head trained above is reused unchanged, so the model that produces the heatmaps
is the model that produced the accuracy.

In [ ]:
#@title 5 - Build the end-to-end model that exposes both feature maps
inp   = Input(shape=(S, S, 3), name="image_uint8_range")
fmaps, pooled = [], []
for name in BB:
    Base, prep = BACKBONES[name]
    net = nets[name]
    x  = Lambda(prep, name="prep_%s" % name)(inp)
    fm = net(x)                                   # (None, 7, 7, C) - last conv block
    fmaps.append(fm)
    pooled.append(GlobalAveragePooling2D(name="gap_%s" % name)(fm))

prob = head(pooled)
full = Model(inp, prob, name="MWAMNet_end_to_end")
grad_model = Model(inp, fmaps + [prob], name="MWAMNet_gradcam")

print("feature maps exposed to Grad-CAM:")
for name, fm in zip(BB, fmaps):
    print("   %-18s %s" % (name, tuple(int(v) for v in fm.shape[1:])))

chk = full.predict(X_te[:64].astype("float32"), verbose=0).ravel()
print("end-to-end model agrees with the cached-feature head on 64 images:",
      bool(np.allclose(chk, p_te[:64], atol=1e-4)))

feature maps exposed to Grad-CAM:
   DenseNet201        (7, 7, 1920)
   MobileNetV3Large   (7, 7, 960)
end-to-end model agrees with the cached-feature head on 64 images: True


In [ ]:
#@title 6 - Grad-CAM at the correct layer, and at the wrong one
import matplotlib.cm as cm

def _norm(h):
    h = np.maximum(h, 0)
    return h / (h.max() + 1e-8)

def gradcam(img_u8, branch=0):
    # Gradient of the fire probability w.r.t. the last conv feature map.
    x = tf.convert_to_tensor(img_u8[None].astype("float32"))
    with tf.GradientTape() as tape:
        outs = grad_model(x, training=False)
        fmap, p = outs[branch], outs[-1][:, 0]
    g = tape.gradient(p, fmap)                       # (1,h,w,C)
    w = tf.reduce_mean(g, axis=(1, 2))               # (1,C)
    cam = tf.reduce_sum(fmap[0] * w[0], axis=-1)     # (h,w)
    return _norm(cam.numpy()), float(p.numpy()[0])


early_models = {}
for name in BB:
    net = nets[name]
    convs = [l for l in net.layers
             if "conv" in l.name.lower()
             and oshape(l) is not None and len(oshape(l)) == 4]
    early_models[name] = Model(net.input, convs[0].output, name="early_%s" % name)
    print("%-18s early layer used for the comparison: %-30s %s"
          % (name, convs[0].name, oshape(convs[0])[1:]))

def gradcam_early(img_u8, name):
    Base, prep = BACKBONES[name]
    em = early_models[name]
    x = tf.convert_to_tensor(img_u8[None].astype("float32"))
    with tf.GradientTape() as tape:
        fmap = em(prep(x), training=False)
        tape.watch(fmap)
        score = tf.reduce_mean(fmap)
    g = tape.gradient(score, fmap)
    w = tf.reduce_mean(g, axis=(1, 2))
    cam = tf.reduce_sum(fmap[0] * w[0], axis=-1)
    return _norm(cam.numpy())

def overlay(ax, img_u8, cam, title, alpha=0.45):
    h = np.array(Image.fromarray((cam * 255).astype("uint8")).resize((S, S), Image.BILINEAR)) / 255.0
    ax.imshow(img_u8)
    ax.imshow(cm.jet(h)[..., :3], alpha=alpha)
    ax.set_title(title, fontsize=9)
    ax.axis("off")

# pick informative examples from the held-out set
pred = (p_te > 0.5).astype(int)
fire_ok   = np.where((y_te == 1) & (pred == 1))[0]
nofire_ok = np.where((y_te == 0) & (pred == 0))[0]
wrong     = np.where(pred != y_te)[0]
print("held-out: %d correct fires, %d correct no-fires, %d errors"
      % (len(fire_ok), len(nofire_ok), len(wrong)))

DenseNet201        early layer used for the comparison: conv1_conv                     (112, 112, 64)
MobileNetV3Large   early layer used for the comparison: conv                           (112, 112, 16)
held-out: 189 correct fires, 187 correct no-fires, 4 errors


In [ ]:
#@title 6a - EVIDENCE FIGURE: early layer vs last layer
i = int(fire_ok[0]) if len(fire_ok) else 0
img = X_te[i]
cam_last_d, p = gradcam(img, 0)
cam_last_m, _ = gradcam(img, 1)
cam_early_d   = gradcam_early(img, "DenseNet201")

fig, ax = plt.subplots(1, 4, figsize=(11, 3.1))
ax[0].imshow(img)
ax[0].set_title("Input (true: %s)" % ("fire" if y_te[i] == 1 else "no-fire"), fontsize=9)
ax[0].axis("off")
overlay(ax[1], img, cam_early_d,
        "First conv layer (%dx%d)\nas in the submitted figures"
        % (cam_early_d.shape[0], cam_early_d.shape[1]))
overlay(ax[2], img, cam_last_d,
        "Last conv layer, DenseNet201 (%dx%d)" % cam_last_d.shape)
overlay(ax[3], img, cam_last_m,
        "Last conv layer, MobileNetV3Large (%dx%d)" % cam_last_m.shape)
fig.suptitle("Grad-CAM layer choice. p(fire) = %.3f" % p, fontsize=10)
fig.tight_layout()
for ext in ("pdf", "png"):
    fig.savefig(os.path.join(OUT, "gradcam_layer_comparison." + ext))
plt.close(fig)
print("saved gradcam_layer_comparison.pdf / .png")

saved gradcam_layer_comparison.pdf / .png


In [ ]:
#@title 6b - REPLACEMENT FIGURE: Grad-CAM panels for the paper

def take(arr, n, label):
    return [(int(i), label) for i in np.asarray(arr)[:n]]

picks  = take(fire_ok, 2, "fire, correct")
picks += take(nofire_ok, 1, "no-fire, correct")
picks += take(wrong, 1, "MISCLASSIFIED")
if not picks:
    picks = [(i, "test image") for i in range(min(3, len(y_te)))]
print("rows in the figure:", [lab for _, lab in picks])

rows = len(picks)
fig, ax = plt.subplots(rows, 3, figsize=(8.2, 2.7 * rows))
ax = np.atleast_2d(ax)
for r, (i, lab) in enumerate(picks):
    img = X_te[i]
    cd, p = gradcam(img, 0)
    cm_, _ = gradcam(img, 1)
    truth = "fire" if y_te[i] == 1 else "no-fire"
    ax[r, 0].imshow(img); ax[r, 0].axis("off")
    ax[r, 0].set_title("%s\ntrue: %s   p(fire) = %.3f" % (lab, truth, p), fontsize=9)
    overlay(ax[r, 1], img, cd,  "DenseNet201 branch")
    overlay(ax[r, 2], img, cm_, "MobileNetV3Large branch")
fig.suptitle("Grad-CAM at the last convolutional layer of each backbone,\n"
             "held-out test set", fontsize=10)
fig.tight_layout(rect=[0, 0, 1, 0.97])
for ext in ("pdf", "png"):
    fig.savefig(os.path.join(OUT, "gradcam_panels." + ext))
plt.close(fig)
print("saved gradcam_panels.pdf / .png   (%d rows, including %d error case)"
      % (rows, 1 if len(wrong) else 0))

rows in the figure: ['fire, correct', 'fire, correct', 'no-fire, correct', 'MISCLASSIFIED']
saved gradcam_panels.pdf / .png   (4 rows, including 1 error case)


## 7 · SHAP

`shap.Explainer` with the `inpaint_telea` masker, as the manuscript describes, run
against the end-to-end model so the attributions correspond to the same predictions.

In [ ]:
#@title 7 - SHAP attributions
try:
    import shap
    def f(x):
        return full.predict(x.astype("float32"), verbose=0)

    masker = shap.maskers.Image("inpaint_telea", (S, S, 3))
    explainer = shap.Explainer(f, masker, output_names=["p(fire)"])

    sel = [int(a[0]) for a in (fire_ok, nofire_ok) if len(a)] or [0]
    sv = explainer(X_te[sel].astype("float32"),
                   max_evals=1200, batch_size=32,
                   outputs=shap.Explanation.argsort.flip[:1])

    fig, ax = plt.subplots(len(sel), 2, figsize=(6.6, 3.0 * len(sel)))
    ax = np.atleast_2d(ax)
    for r, i in enumerate(sel):
        vals = sv.values[r]
        if vals.ndim == 4: vals = vals[..., 0]
        m = np.abs(vals).sum(-1)
        m = m / (m.max() + 1e-8)
        ax[r, 0].imshow(X_te[i]); ax[r, 0].axis("off")
        ax[r, 0].set_title("Input (true: %s)" % ("fire" if y_te[i] else "no-fire"), fontsize=9)
        ax[r, 1].imshow(X_te[i])
        ax[r, 1].imshow(m, cmap="jet", alpha=0.45)
        ax[r, 1].set_title("SHAP attribution magnitude", fontsize=9)
        ax[r, 1].axis("off")
    fig.suptitle("SHAP pixel attributions, held-out test set", fontsize=10)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(OUT, "shap_panels." + ext))
    plt.close(fig)
    print("saved shap_panels.pdf / .png")
except Exception as e:
    print("SHAP step did not complete:", type(e).__name__, e)
    print("This is not fatal - the Grad-CAM figures above are the ones the paper needs.")

  0%|          | 0/1198 [00:00<?, ?it/s]

PartitionExplainer explainer:  50%|█████     | 1/2 [00:00<?, ?it/s]

  0%|          | 0/1198 [00:00<?, ?it/s]

PartitionExplainer explainer: 3it [05:03, 151.79s/it]


saved shap_panels.pdf / .png


In [ ]:
#@title 8 - Zip and download
import shutil
for p in sorted(glob.glob(os.path.join(OUT, "*"))):
    print("%10d  %s" % (os.path.getsize(p), os.path.basename(p)))

shutil.make_archive("/content/MWAMNet_figures", "zip", OUT)
try:
    from google.colab import files
    files.download("/content/MWAMNet_figures.zip")
except Exception as e:
    print("Download it from the file browser on the left:", e)

    486453  gradcam_layer_comparison.pdf
    524597  gradcam_layer_comparison.png
       433  gradcam_layer_diagnosis.json
   4298739  gradcam_panels.pdf
   4567786  gradcam_panels.png
    511865  shap_panels.pdf
    556323  shap_panels.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>